In [1]:
import os
import sys

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [7]:
from src.ab_testing import (
    LLMEvaluator,
    MovieRecommendation,
    RecommendationList,
    UserContext,
)

/home/visuworks2019/Projects/mk_folder/movie_recommendation/src/ab_testing/llm/prompts.py:120: UserWarning: Field name "schema" in "OutputFormatPrompt" shadows an attribute in parent "BaseModel"
  class OutputFormatPrompt(BaseModel):


In [2]:
from data_scraping.common import load_movie_data
from modeling.models.query_search import QuerySearchPipeline

2025-11-25 18:01:43.958348738 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [3]:
# 영화 데이터 로드
movie_data = load_movie_data()
print(f"📊 영화 데이터 로드 완료: {len(movie_data)}개")

# 검색 파이프라인 초기화 및 학습
pipeline = QuerySearchPipeline()
pipeline.fit(movie_data)
print("✅ 검색 인덱스 생성 완료")

📊 영화 데이터 로드 완료: 79093개
✅ 검색 인덱스 생성 완료


In [14]:
import pandas as pd

with pd.option_context("display.max_columns", None):
    display(movie_data.sample(3))

,movie_id,title,genres,imdb_id,tmdb_id,adult,backdrop_path,id,title_tmdb,original_title,overview,poster_path,media_type,original_language,genre_ids,popularity,release_date,video,vote_average,vote_count,genres_tmdb,language,total_title
24165,201436,What We Left Behind: Looking Back at Star Trek...,Documentary,tt6332276,522055.0,False,/ptPwo47G8i5KTE68vAmBzgqWhlG.jpg,522055,What We Left Behind: Looking Back at Star Trek...,What We Left Behind: Looking Back at Star Trek...,,/90kkGkEA3Svwt5SeCaR28pDm5dQ.jpg,movie,en,[99],2.9263,2018-10-12,False,7.0,51.0,다큐멘터리,영어,What We Left Behind: Looking Back at Star Trek...
36406,161088,Born to Ride (2011),Action Thriller,tt1721491,68995.0,False,/pfPuCQybXvacUgtHeFCZ03ouUzF.jpg,68995,Born to Ride,Born to Ride,,/qHlPQ9P6CYiOD8NlJ1wtntcvq88.jpg,movie,en,"[28, 53]",2.8923,2011-06-26,False,3.6,23.0,액션 스릴러,영어,Born to Ride
27872,80984,Rogue Cop (1954),Crime Drama Film-Noir,tt0047424,30036.0,False,/jhGkvf0HERYiJ0X4BaM6ADM7LMH.jpg,30036,Rogue Cop,Rogue Cop,,/kCYHFg6ke5lOb3buIZ38dj9VkF1.jpg,movie,en,"[80, 18]",3.4288,1954-09-17,False,5.3,12.0,범죄 드라마,영어,Rogue Cop


In [44]:
# 검색 쿼리 설정
query = "thriler movie with blood"
top_k = 10

# 검색 실행
results = pipeline.search_to_response(query, top_k=top_k)

In [46]:
# 검색 결과를 DataFrame으로 변환
import pandas as pd

# 결과에서 movie_id만 추출
result_movie_ids = [movie.movie_id for movie in results.results]

# movie_data에서 movie_id로 필터링 (movie_id가 str일 수 있으므로 문자열로 변환 비교)
recommended_movie_data = movie_data[
    movie_data["movie_id"].astype(str).isin(result_movie_ids)
].copy()

# 결과 정렬: 결과의 movie_id 순서와 일치시키기 위해 인덱스 정렬
recommended_movie_data["movie_id"] = recommended_movie_data["movie_id"].astype(str)
recommended_movie_data["search_score"] = [
    next(
        (movie.score for movie in results.results if str(movie.movie_id) == movie_id),
        None,
    )
    for movie_id in recommended_movie_data["movie_id"]
]
filtered_movie_data = (
    recommended_movie_data.set_index("movie_id").loc[result_movie_ids].reset_index()
)

print("📊 검색 결과 DataFrame:")
filtered_movie_data.shape[0]

📊 검색 결과 DataFrame:


10

In [47]:
# filtered_movie_data에서 추천 리스트 생성
list_a = RecommendationList(
    list_id="A",
    system_name="System A",
    recommendations=[
        MovieRecommendation(
            movie_id=row["movie_id"],
            title=row["title"],
            year=(
                int(row["release_date"])
                if "year" in row and pd.notnull(row["year"])
                else None
            ),
            genres=(
                [g.strip() for g in str(row["genres"]).split()]
                if "genres" in row
                else []
            ),
            description=row["overview"] if "overview" in row else "",
        )
        for _, row in recommended_movie_data.iterrows()
    ],
)

genres:
- SF
- TV
- 가족
- 공포
- 다큐멘터리
- 드라마
- 로맨스
- 모험
- 미스터리
- 범죄
- 서부
- 스릴러
- 애니메이션
- 액션
- 역사
- 영화
- 음악
- 전쟁
- 코미디
- 판타지

In [48]:
# 필터링 옵션을 사용한 검색 예제
results_filtered = pipeline.search_to_response(
    query=query,
    top_k=10,
    min_score=5.0,  # 최소 스코어
    min_rating=7.0,  # 최소 평점
    min_vote_count=100,  # 최소 평가 수
    genre_filter=["스릴러"],  # 장르 필터
)
results_filtered

QuerySearchResponse(query='thriler movie with blood', total_results=10, results=[SearchResultMovie(movie_id='2403', title='람보 (First Blood)', genres='액션 모험 스릴러 전쟁', score=18.96830791545386, overview='월남전에서 제대한 그린베레 출신의 존 람보(실베스타 스탤론 분)는, 전우를 찾아 록키 산맥의 어느 한적한 시골 마을에 도착한다. 하지만 그가 찾고자 하는 이는 이미 암으로 세상을 떠난 뒤였다. 마을 보안관 셔리프 윌 티즐(브라이언 데니히 분)는 그의 부랑자 행색에 반감을 갖고 마을에서 쫓아내려한다. 하지만 람보가 순순히 응하지 않자 억지 죄목으로 체포하는데, 조사를 하는 과정에서 그 옛날 월맹 포로 수용소에서 받은 고문 기억이 악몽처럼 되살아나자, 람보는 갑자기 미친 사람처럼 광폭해져 경찰관과 경찰서를 때려 부수고 탈출한다. 경찰의 추적을 따돌리고 마을 산 속으로 숨어든 람보는 월남전에서 몸에 익힌 게릴라 전술로 경찰과 대치한다. 그러다 경찰 헬기를 피하고자 절벽에서 뛰어내려 결국 큰 부상을 입는다. 이때 저격의 위험 때문에 그가 던진 돌이 헬기 유리창을 맞추고 이틈에 경찰관 하트가 헬기에서 떨어져 죽는다. 위험에서 벗어난 람보는 피가 흐르는 팔을 바늘로 꿰맨다. 동료마저 잃은 경찰은 람보를 잡는데 혈안이 되어 사냥개를 이끌고 숲으로 추적해 오지만 결국 람보의 교묘한 전술로 모두 부상을 입고 물러난다. 사태가 커지자 지방 경찰 기동대와 주경비대대의 지원을 받아 주변의 모든 길이 통제되고, 이 일은 신문과 방송 등 메스컴의 집중을 받는데...', matched_fields={'title': 18.96830791545386}, year=None, cast_info=None), SearchResultMovie(movie_id='170799', title='트윈 픽스: 더 미씽 피시즈 (Twin Peaks: The 

In [49]:
# 검색 결과를 DataFrame으로 변환
import pandas as pd

# 결과에서 movie_id만 추출
result_movie_ids = [movie.movie_id for movie in results_filtered.results]

# movie_data에서 movie_id로 필터링 (movie_id가 str일 수 있으므로 문자열로 변환 비교)
filtered_recommended_movie_data = movie_data[
    movie_data["movie_id"].astype(str).isin(result_movie_ids)
].copy()

# 결과 정렬: 결과의 movie_id 순서와 일치시키기 위해 인덱스 정렬
filtered_recommended_movie_data["movie_id"] = filtered_recommended_movie_data[
    "movie_id"
].astype(str)
filtered_recommended_movie_data["search_score"] = [
    next(
        (
            movie.score
            for movie in results_filtered.results
            if str(movie.movie_id) == movie_id
        ),
        None,
    )
    for movie_id in filtered_recommended_movie_data["movie_id"]
]
filtered_recommended_movie_data = (
    filtered_recommended_movie_data.set_index("movie_id")
    .loc[result_movie_ids]
    .reset_index()
)

print("📊 검색 결과 DataFrame:")
filtered_recommended_movie_data.shape[0]

📊 검색 결과 DataFrame:


10

In [50]:
# filtered_movie_data에서 추천 리스트 생성
list_b = RecommendationList(
    list_id="B",
    system_name="System A",
    recommendations=[
        MovieRecommendation(
            movie_id=row["movie_id"],
            title=row["title"],
            year=(
                int(row["release_date"])
                if "year" in row and pd.notnull(row["year"])
                else None
            ),
            genres=(
                [g.strip() for g in str(row["genres"]).split()]
                if "genres" in row
                else []
            ),
            description=row["overview"] if "overview" in row else "",
        )
        for _, row in filtered_recommended_movie_data.iterrows()
    ],
)

In [51]:
evaluator = LLMEvaluator()

In [52]:
sci_fi_context = UserContext(
    user_description=(
        "A sci-fi enthusiast who loves mind-bending plots, visual effects, and thought-provoking narratives. Age 25-35."
    )
)

thriller_dark_context = UserContext(
    user_description=(
        "Loves thriller and dark-themed movies, enjoys intense suspense and mysterious atmospheres. Prefers movies with twists and psychological depth. Age 28-40."
    )
)

In [53]:
result = evaluator.evaluate_lists(sci_fi_context, list_a, list_b)

print("\n" + "=" * 80)
print("📊 평가 결과")
print("=" * 80)
print(f"\n선호 리스트: {result.preferred_list}")
print("\n이유:")
print(f"  {result.reasoning}")
print("\n클릭한 영화 ID 목록:")
print(
    f"  🎬 List A (System A): {result.clicked_item_ids_A if result.clicked_item_ids_A else '없음'}"
)
print(
    f"  🎬 List B (System B): {result.clicked_item_ids_B if result.clicked_item_ids_B else '없음'}"
)


📊 평가 결과

선호 리스트: B

이유:
  List B includes 'Patlabor 2: The Movie' (id=132362), a sci-fi action film with animation, which directly matches the user's sci-fi preference. While List A focuses on horror/thrillers, it lacks sci-fi elements. List B also features 'Girl with the Dragon Tattoo, The' (id=91658) with a complex plot, aligning with the user's love for mind-bending narratives. List A's horror-centric titles (e.g., 'Blood Honey', 'Blood Sabbath') may not appeal to the user's sci-fi focus.

클릭한 영화 ID 목록:
  🎬 List A (System A): 없음
  🎬 List B (System B): ['132362', '91658']


In [43]:
result = evaluator.evaluate_lists(thriller_dark_context, list_a, list_b)

print("\n" + "=" * 80)
print("📊 평가 결과")
print("=" * 80)
print(f"\n선호 리스트: {result.preferred_list}")
print("\n이유:")
print(f"  {result.reasoning}")
print("\n클릭한 영화 ID 목록:")
print(
    f"  🎬 List A (System A): {result.clicked_item_ids_A if result.clicked_item_ids_A else '없음'}"
)
print(
    f"  🎬 List B (System B): {result.clicked_item_ids_B if result.clicked_item_ids_B else '없음'}"
)


📊 평가 결과

선호 리스트: B

이유:
  List B aligns better with the user's preference for thrillers, suspense, and dark themes. Movies like 'Seven Sisters', 'Timecrimes', and 'Limitless' offer psychological depth, sci-fi elements, and twists. List A is dominated by comedies and lacks thriller elements.

클릭한 영화 ID 목록:
  🎬 List A (System A): ['26290']
  🎬 List B (System B): ['173925', '65642', '84152', '6979', '3527']


# 쿼리 생성 

In [1]:
import os
import sys

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from src.ab_testing import QueryGenerator, UserContext

/home/visuworks2019/Projects/mk_folder/movie_recommendation/src/ab_testing/prompts/evaluation.py:83: UserWarning: Field name "schema" in "OutputFormatPrompt" shadows an attribute in parent "BaseModel"
  class OutputFormatPrompt(BaseModel):


In [3]:
user_context = UserContext(
    user_description=(
        "A sci-fi enthusiast who loves mind-bending plots, "
        "visual effects, and thought-provoking narratives. Age 25-35."
    )
)

In [4]:
generator = QueryGenerator()

result = generator.generate_query(
    user_context=user_context,
    service_type="natural_language_search",
)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
result.query

'Mind-bending sci-fi with stunning visuals'